# Téléchargement du réseau piéton - Canton de Genève

Ce notebook permet de télécharger et analyser les données de réseau piéton du Canton de Genève depuis OpenStreetMap, en utilisant les classes spécifiques à la marche (pedestrian, footway, steps, etc.).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
sys.path.append(os.path.abspath('../../'))

# Imports
import geopandas as gpd
import osmnx as ox
import pandas as pd
import matplotlib.pyplot as plt
import folium
from shapely.geometry import Point, LineString
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

print("📦 Libraries importées avec succès !")

In [ ]:
operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

input_file_path = '../../Data/input'
output_step1_path='../../Data/output/step-1'

## 1. Définition de la zone d'étude

In [ ]:
%%time
# Récupérer les limites du Canton de Genève depuis OSM
print("🗺️ Téléchargement des limites du Canton de Genève...")

canton_geneve = ox.geocode_to_gdf("Canton de Genève, Switzerland")
print("✅ Canton de Genève trouvé")

print(f"📐 Superficie: {canton_geneve.to_crs('EPSG:2056').area.iloc[0]/1e6:.1f} km²")
canton_geneve.head()

In [ ]:
# # Import manquant pour la bounding box
# from shapely.geometry import box

# # Visualisation rapide de la zone
# m_canton = folium.Map(location=[46.25, 6.15], zoom_start=10, tiles='CartoDB Positron')

# # Ajouter les limites du canton
# folium.GeoJson(
#     canton_geneve.__geo_interface__,
#     style_function=lambda x: {
#         'fillColor': '#3489db',
#         'color': '#E60000',
#         'weight': 2,
#         'fillOpacity': 0.1,
#     }
# ).add_to(m_canton)

# print("🗺️ Carte des limites créée (exécuter m_canton pour l'afficher)")
# m_canton  # Décommenter pour afficher

## 2. Téléchargement du réseau piéton

Nous allons télécharger spécifiquement les infrastructures dédiées à la marche selon les classifications OpenStreetMap.

In [ ]:
import os
import osmnx as ox
import geopandas as gpd
import pandas as pd

# =========================================================
# 0) PARAMÈTRES (assume canton_geneve déjà chargé)
# =========================================================
polygon = canton_geneve.geometry.iloc[0]

# Télécharger "large" côté highway pour filtrer en local
custom_filter = (
    '["area"!~"yes"]'
    '["highway"!~"motorway|motorway_link|trunk|trunk_link|construction|proposed"]'
    '["highway"~"footway|path|pedestrian|steps|living_street|residential|service|unclassified|tertiary|secondary|primary|platform"]'
)

# Options de comportement
PROMOTE_SECONDARY_WITH_SIDEWALK = False  # promouvoir secondaires avec trottoirs explicites → prioritaire
ASSUME_FOOT_OK_ON_LOCAL       = True    # hypothèse: foot OK si non renseigné sur routes locales
LOCAL_ROADS = {"residential", "tertiary", "service","unclassified"}  # tu peux ajouter "tertiary" si souhaité

# =========================================================
# 1) TÉLÉCHARGER LE GRAPHE + GDF
# =========================================================
print("🌍 Téléchargement du réseau piéton élargi…")
G = ox.graph_from_polygon(
    polygon=polygon,
    custom_filter=custom_filter,
    simplify=True,
    retain_all=True
)

print("🔄 Conversion en GeoDataFrames…")
nodes_gdf = ox.graph_to_gdfs(G, edges=False)
edges_gdf = ox.graph_to_gdfs(G, nodes=False, fill_edge_geometry=True).reset_index(drop=True)

print(f"📊 Nœuds: {len(nodes_gdf)} | Arêtes: {len(edges_gdf)}")
print("📋 Colonnes des arêtes:", list(edges_gdf.columns))

# =========================================================
# 2) HELPERS (highway en listes, sidewalks, foot, platforms)
# =========================================================
SIDEWALK_OK   = {"both","left","right","yes","separate"}  # 'separate' = trottoirs cartographiés à part
FOOT_POSITIVE = {"yes","designated","permissive"}
FOOT_NEGATIVE = {"no","private","use_sidepath"}

def _lower(x): 
    return str(x).lower() if x is not None else ""

def _as_set(v):
    """Convertit highway en set (gère liste/tuple/NaN)."""
    if isinstance(v, (list, tuple, set)):
        return { _lower(e) for e in v }
    return { _lower(v) }

def has_sidewalk(row):
    vals = { _lower(row.get("sidewalk")),
             _lower(row.get("sidewalk:left")),
             _lower(row.get("sidewalk:right")) }
    return len(SIDEWALK_OK & vals) > 0

def foot_allowed_generic(row):
    f, acc = _lower(row.get("foot")), _lower(row.get("access"))
    if f in FOOT_NEGATIVE or acc in {"no","private"}:
        return False
    if f in FOOT_POSITIVE:
        return True
    # indéterminé → hypothèse sur routes locales
    if ASSUME_FOOT_OK_ON_LOCAL:
        hw = _as_set(row.get("highway"))
        if len(LOCAL_ROADS & hw) > 0:
            return True
    return False

def foot_allowed_on_path(row):
    # Politique 'path' : autorisé par défaut sauf interdiction explicite
    f, acc = _lower(row.get("foot")), _lower(row.get("access"))
    if f in {"no","private"} or acc in {"no","private"}:
        return False
    return True

def any_tag_in(hw_set, wanted):
    return any(tag in hw_set for tag in wanted)

def get_platforms_from_polygon(poly, tags):
    """Getter version-agnostic OSMnx pour features/geometries_from_polygon; fallback bbox+clip."""
    try:
        return ox.features_from_polygon(poly, tags=tags)
    except AttributeError:
        pass
    try:
        return ox.geometries_from_polygon(poly, tags=tags)
    except AttributeError:
        pass
    minx, miny, maxx, maxy = poly.bounds
    north, south, east, west = maxy, miny, maxx, minx
    gdf_bbox = ox.geometries_from_bbox(north, south, east, west, tags=tags)
    try:
        return gpd.clip(gdf_bbox, poly)
    except Exception:
        return gdf_bbox[gdf_bbox.geometry.intersects(poly)]

# =========================================================
# 3) 🎯 PRIORITAIRE / SECONDAIRE (PATCH COMPLÉMENTAIRE)
# =========================================================
print("🎯 Filtrage PRIORITAIRE / SECONDAIRE…")

edges_norm = edges_gdf.copy()
edges_norm["hw_set"] = edges_norm["highway"].apply(_as_set)

# PRIORITAIRE
# - footway/pedestrian/steps/living_street
# - path (autorisé par défaut sauf interdiction explicite)
# - routes locales (LOCAL_ROADS) AVEC trottoir explicite
priority_mask = (
    edges_norm["hw_set"].apply(lambda s: any(t in s for t in ["footway","pedestrian","steps","living_street"]))
    | (edges_norm["hw_set"].apply(lambda s: "path" in s) & edges_norm.apply(foot_allowed_on_path, axis=1))
    | (edges_norm["hw_set"].apply(lambda s: len(LOCAL_ROADS & s) > 0) & edges_norm.apply(has_sidewalk, axis=1))
)

priority_pedestrian = edges_norm[priority_mask].copy()
priority_pedestrian["tier"] = "priority"

# SECONDAIRE
# - routes locales SANS trottoir explicite mais marche autorisée (ou supposée OK)
secondary_mask = (
    edges_norm["hw_set"].apply(lambda s: len(LOCAL_ROADS & s) > 0)
    & (~edges_norm.apply(has_sidewalk, axis=1))
    & (edges_norm.apply(foot_allowed_generic, axis=1))
)
secondary_pedestrian = edges_norm[secondary_mask].copy()
secondary_pedestrian["tier"] = "secondary"

# (Option) promotion des secondaires avec trottoir explicite → prioritaire
if PROMOTE_SECONDARY_WITH_SIDEWALK:
    sw_idx = secondary_pedestrian[secondary_pedestrian.apply(has_sidewalk, axis=1)].index
    if len(sw_idx) > 0:
        promoted = secondary_pedestrian.loc[sw_idx].copy()
        promoted["tier"] = "priority"
        secondary_pedestrian = secondary_pedestrian.drop(index=sw_idx)
        priority_pedestrian = gpd.GeoDataFrame(
            pd.concat([priority_pedestrian, promoted], ignore_index=True),
            geometry="geometry", crs=edges_norm.crs
        )

print(f"🚶 PRIORITAIRE (avant arrêts): {len(priority_pedestrian)}")
print(f"🏘️ SECONDAIRE: {len(secondary_pedestrian)}")

# =========================================================
# 4) ➕ PLATEFORMES (bus/tram/train) ajoutées au PRIORITAIRE
# =========================================================
platform_tags = {
    "highway": "platform",
    "public_transport": "platform",
    "railway": "platform",
    "amenity": "bus_station",
}
plats = get_platforms_from_polygon(polygon, tags=platform_tags)
print(f"🚏 Objets plateformes bruts: {0 if plats is None else len(plats)}")

if plats is not None and not plats.empty:
    plats = plats.to_crs(edges_gdf.crs)
    parts = []
    p_lines = plats[plats.geometry.geom_type.isin(["LineString","MultiLineString"])].copy()
    if not p_lines.empty:
        parts.append(p_lines[["geometry"]])
    p_poly = plats[plats.geometry.geom_type.isin(["Polygon","MultiPolygon"])].copy()
    if not p_poly.empty:
        p_poly["geometry"] = p_poly.geometry.boundary
        parts.append(p_poly[["geometry"]])

    platforms_edges = gpd.GeoDataFrame(
        pd.concat(parts, ignore_index=True) if parts else pd.DataFrame({"geometry":[]}),
        geometry="geometry", crs=edges_gdf.crs
    )
    if not platforms_edges.empty and platforms_edges.crs and platforms_edges.crs.is_projected:
        platforms_edges = platforms_edges[platforms_edges.length > 0.5]  # > 0.5 m
    platforms_edges["tier"] = "priority"
    platforms_edges["source"] = "platform"
else:
    platforms_edges = gpd.GeoDataFrame(geometry=[], crs=edges_gdf.crs)

print(f"🚏 Plateformes en lignes: {len(platforms_edges)}")

# Snap léger des plateformes vers le PRIORITAIRE, puis concat
PLATFORM_SNAP_M = 5.0
if len(platforms_edges) > 0 and len(priority_pedestrian) > 0:
    sindex = priority_pedestrian.sindex
    snapped = []
    for g in platforms_edges.geometry:
        try:
            idx = list(sindex.nearest(g.bounds, 1))[0]
            nearest = priority_pedestrian.iloc[idx].geometry
            snapped.append(nearest if g.distance(nearest) <= PLATFORM_SNAP_M else g)
        except Exception:
            snapped.append(g)
    platforms_edges = platforms_edges.copy()
    platforms_edges["geometry"] = snapped

    priority_pedestrian = gpd.GeoDataFrame(
        pd.concat([priority_pedestrian, platforms_edges.reindex(columns=priority_pedestrian.columns, fill_value=None)],
                  ignore_index=True),
        geometry="geometry", crs=edges_gdf.crs
    )

print(f"✅ PRIORITAIRE (avec arrêts): {len(priority_pedestrian)}")
print(f"✅ SECONDAIRE: {len(secondary_pedestrian)}")

# =========================================================
# 5) ASSEMBLAGE GLOBAL (tier dans la colonne 'tier')
# =========================================================
walkable_all = pd.concat([priority_pedestrian, secondary_pedestrian], ignore_index=True)
walkable_all = gpd.GeoDataFrame(walkable_all, geometry="geometry", crs=edges_gdf.crs)

print(f"🌐 Réseau marchable total: {len(walkable_all)}")
print("📌 Colonnes:", list(walkable_all.columns))

# =========================================================
# 6) EXPORT
# =========================================================
try:
    input_file_path
except NameError:
    input_file_path = "./data"

output_dir = f"{input_file_path}/network"
os.makedirs(output_dir, exist_ok=True)
print(f"📁 Dossier d'export: {output_dir}")

def _export(gdf, name):
    if "export_data" in globals():
        return export_data(gdf, name, formats=["geoparquet","geojson","pickle"])
    else:
        paths = []
        gpq = os.path.join(output_dir, f"{name}.geoparquet")
        gj  = os.path.join(output_dir, f"{name}.geojson")
        gdf.to_parquet(gpq)
        gdf.to_file(gj, driver="GeoJSON")
        paths.extend([gpq, gj])
        return paths

print("💾 Export PRIORITAIRE…")
print(_export(priority_pedestrian, "walkable_priority"))

print("💾 Export SECONDAIRE…")
print(_export(secondary_pedestrian, "walkable_secondary"))

print("💾 Export GLOBAL…")
print(_export(walkable_all, "walkable_all"))

print("✅ Terminé.")

🌍 Téléchargement du réseau piéton élargi…
🔄 Conversion en GeoDataFrames…
📊 Nœuds: 84979 | Arêtes: 210210
📋 Colonnes des arêtes: ['osmid', 'highway', 'lanes', 'maxspeed', 'name', 'ref', 'oneway', 'reversed', 'length', 'geometry', 'tunnel', 'access', 'service', 'junction', 'width', 'bridge', 'area']
🎯 Filtrage PRIORITAIRE / SECONDAIRE…
🚶 PRIORITAIRE (avant arrêts): 95425
🏘️ SECONDAIRE: 56544
🚏 Objets plateformes bruts: 1931
🚏 Plateformes en lignes: 1608
✅ PRIORITAIRE (avec arrêts): 97033
✅ SECONDAIRE: 56544
🌐 Réseau marchable total: 153577
📌 Colonnes: ['osmid', 'highway', 'lanes', 'maxspeed', 'name', 'ref', 'oneway', 'reversed', 'length', 'geometry', 'tunnel', 'access', 'service', 'junction', 'width', 'bridge', 'area', 'hw_set', 'tier']
📁 Dossier d'export: ../../Data/input/network
💾 Export PRIORITAIRE…
❌ Erreur geoparquet: ('cannot mix list and non-list, non-null values', 'Conversion failed for column osmid with type object')
✅ GEOJSON: ../../Data/input/network/walkable_priority.geojson
